## Transformer Block and Attention Mechanisms

0. Make sure all the dimensions follow the main logic of the matrix [X]
1. Add the parameters for the tensor dimensions for the attention block. [X]
2. Add the masking matrix to the self-attention block 
3. Double check that all the multiplications go as planned. I recieve the expected outcomes.

### Attention Mechanisms

In [4]:
import math
import torch
import torch.nn as nn

D = 6 # number of dimensions (for the embeddings as far as i am concerned)
I = 5 # number of input tokens (as far as i am concerned
N_ATT = 2

X = torch.rand((D, I)) # Dimensions are rows, and I-inputs are individual tokens

In [5]:
class SelfAttention_Block(nn.Module):    
    def __init__(self, num_att_heads: int, D_dimension: int):
        super().__init__()
        self.D = D_dimension
        # linear projections  W_*  ∈ ℝ[D × D]
        self.Q_w = nn.Parameter(torch.randn(self.D, self.D))
        self.K_w = nn.Parameter(torch.randn(self.D, self.D))
        self.V_w = nn.Parameter(torch.randn(self.D, self.D))

        self.Q_b = nn.Parameter(torch.randn(self.D, 1))
        self.K_b = nn.Parameter(torch.randn(self.D, 1))
        self.V_b = nn.Parameter(torch.randn(self.D, 1))

        self.softmax = nn.Softmax(dim=1) 

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # 1) linear projections
        Q = self.Q_w @ X + self.Q_b            # (D, I)
        K = self.K_w @ X + self.K_b
        V = self.V_w @ X + self.V_b

        # 2) scaled dot-product attention
        attn = (Q.t() @ K) / math.sqrt(self.D)         # (I, I)
        attn   = self.softmax(attn)                      # (I, I)

        # 3) weighted sum of values for each query token
        context = (attn @ V.t()).t()                       # (D, I)
        return context

In [6]:
block = SelfAttention_Block(1, D)
out   = block(X)           # forward

loss  = out.sum()          # scalar, any criterion ok
block.zero_grad()          # clear
loss.backward()            # backward pass

print(block.K_w.grad)   # leaf param grad
print(block.V_w.grad)     # non-leaf grad (only if you saved it)

tensor([[-0.1348, -0.0070, -0.0124,  0.0505, -0.1156, -0.0625],
        [-0.3100, -0.0230, -0.0584,  0.1333, -0.2584, -0.0528],
        [-0.7617, -0.0417, -0.1122,  0.3095, -0.6339, -0.2312],
        [ 0.4677,  0.0196,  0.0610, -0.1858,  0.3864,  0.1697],
        [-0.3056, -0.0144, -0.0460,  0.1246, -0.2511, -0.0920],
        [-0.0545,  0.0023, -0.0076,  0.0220, -0.0395, -0.0223]])
tensor([[3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096],
        [3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096],
        [3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096],
        [3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096],
        [3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096],
        [3.1409, 2.0076, 1.9921, 1.5638, 2.7025, 3.6096]])


In [7]:
class MultiHead_SelfAttention_Block(nn.Module):    
    def __init__(self, num_att_heads: int, D_dimension: int):
        super().__init__()
        assert D_dimension % num_att_heads == 0, "D must be divisible by the number of heads"
        self.heads_n = num_att_heads
        self.D = D_dimension
        self.d_k = D_dimension // self.heads_n            # head size

        self.Q_w = nn.ParameterList()
        self.K_w = nn.ParameterList()
        self.V_w = nn.ParameterList()

        self.Q_b = nn.ParameterList()
        self.K_b = nn.ParameterList()
        self.V_b = nn.ParameterList()

        self.O_w = nn.Parameter(torch.randn(self.D, self.D))
        # linear projections  W_*  ∈ ℝ[d_k × D]
        for i in range(num_att_heads):
            self.Q_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))
            self.K_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))
            self.V_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))

            self.Q_b.append(nn.Parameter(torch.randn(self.d_k, 1)))
            self.K_b.append(nn.Parameter(torch.randn(self.d_k, 1)))
            self.V_b.append(nn.Parameter(torch.randn(self.d_k, 1)))

        self.softmax = nn.Softmax(dim=1) 

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        head_contexts = []

        for i in range(self.heads_n):
            # 1) linear projections
            Q = self.Q_w[i] @ X + self.Q_b[i]            # (d_k, I)
            K = self.K_w[i] @ X + self.K_b[i]
            V = self.V_w[i] @ X + self.V_b[i]

            # 2) scaled dot-product attention
            attn = (Q.t() @ K) / math.sqrt(self.d_k)         # (I, I)
            attn   = self.softmax(attn)                      # (I, I)

            # 3) weighted sum of values for each query token
            context = (attn @ V.t()).t()                       # (d_k, I)
            head_contexts.append(context)  
        
        context = torch.cat(head_contexts, dim=0)  # (D, I)
        output = self.O_w @ context  # (D, I)
        return output

In [8]:
block = MultiHead_SelfAttention_Block(N_ATT, D)
out   = block(X)           # forward

loss  = out.sum()          # scalar, any criterion ok
block.zero_grad()          # clear
loss.backward()            # backward pass

print(block.K_w[0].grad)   # leaf param grad
print(block.O_w.grad)     # non-leaf grad (only if you saved it)

tensor([[ 0.3171, -0.3322,  0.3360, -0.2035, -0.3406, -0.4628],
        [ 0.4878, -0.5030,  0.5099, -0.3117, -0.5098, -0.6981],
        [-0.0278,  0.0351, -0.0352,  0.0202,  0.0404,  0.0521]])
tensor([[ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462],
        [ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462],
        [ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462],
        [ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462],
        [ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462],
        [ 7.1726,  2.2194, -2.2216, -8.7357,  2.9789,  3.1462]])


### Tokenization Layer

In [9]:
### implement a tokenazation procedure here

### Transformer Layer

In [ ]:
class FCN_Layer(nn.Module):
    def __init__(self, WHICH_PAREMETERS_ADD_HERE):
        self.W = nn.Linear()
        

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return X

In [ ]:
class Transfromer(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.I_dimensions = I
        self.multi_head_att = MultiHead_SelfAttention_Block(N_ATT, D)
        self.layer_norm_1 = nn.LayerNorm(normalized_shape=(D, I))
        self.layer_norm_2 = nn.LayerNorm(normalized_shape=(D, I))
        self.fcn_layers = nn.ParameterList()

        for i in range(self.I_dimensions):
            self.fcn_layers.append(FCN_Layer())

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        X = X + self.multi_head_att(X)
        X = self.layer_norm_1(X)
        x = []
        for i in range(self.I_dimensions):
            x.append(self.fcn_layers[i](X[:, i]) + X[:, i])
        X = torch.concat(x)
        return self.layer_norm_2
